In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_03_crm_events
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_03_crm_events
# Spec Ref  : §1.3 CRM Events Mapping (Salesforce Task → Standardized Events)
# Purpose   : Read from hgi.silver.tasks WHERE source = 'salesforce' AND object = 'Task'
#             Classify Subject field → meta_event vocabulary
#             Output: hgi.silver.crm_events
#
# Key spec requirement:
#   meta_event must be 100% non-null (fallback = cleaned subject string)
#   contact_id built from WhoId/WhatId already resolved in bronze layer
#
# Inputs  : hgi.silver.tasks  (written by 02a_silver_task)
# Output  : hgi.silver.crm_events
#
# Run after: map_01 (contacts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)

print(f"=== Map 03: crm_events (SF Tasks → standardized events) ===  customer={customer_id}  tenant={tenant_id}")
print(f"  Input  : {sv}.tasks")
print(f"  Output : {sv}.crm_events")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"
initialize_audit_tables()

In [0]:
# CELL 3 ── Create crm_events table if not exists
create_map_table(
    f"{sv}.crm_events",
    """
        id              STRING NOT NULL,
        tenant          BIGINT,
        event_timestamp TIMESTAMP,
        contact_id      STRING,
        meta_event      STRING,
        event_text      STRING,
        data_timestamp  TIMESTAMP
    """,
    cluster_by="tenant, meta_event"
)

# Add data_timestamp column if missing (existing table from before)
try:
    existing_cols = [c.name.lower() for c in spark.table(f"{sv}.crm_events").schema]
    if "data_timestamp" not in existing_cols:
        spark.sql(f"ALTER TABLE {sv}.crm_events ADD COLUMN data_timestamp TIMESTAMP")
        print("  Added data_timestamp column to crm_events")
except Exception:
    pass

In [0]:
# CELL 4 ── CDF-Aware Incremental MERGE for crm_events (tenant-scoped)
# FIX v2: Use SOURCE table version (stored as property) instead of TARGET history
# This prevents the "Start version(46) > latest version(26)" error
# Spec §1.3: CASE WHEN subject patterns → meta_event (must be 100% non-null)
try:
    safe_drop_view(f"{sv}.crm_events")

    # Add data_timestamp column if missing (existing table)
    try:
        existing_cols = [c.name.lower() for c in spark.table(f"{sv}.crm_events").schema]
        if "data_timestamp" not in existing_cols:
            spark.sql(f"ALTER TABLE {sv}.crm_events ADD COLUMN data_timestamp TIMESTAMP")
            print("  Added data_timestamp column to crm_events")
    except Exception:
        pass

    # ── VERSION TRACKING (v2): Read last-processed SOURCE version from target property ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.crm_events ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' else 0
    except Exception:
        last_source_ver = 0

    # Get current max version of the SOURCE table (silver.tasks)
    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.tasks)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.tasks  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    # Check if this is first run (crm_events is empty) — do full load
    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.crm_events WHERE tenant = {tenant_id}").collect()[0][0]
    
    # Define the meta_event CASE expression for reuse
    meta_event_case = """
        CASE
            WHEN LOWER(a_subject) = 'call'                          THEN 'call'
            WHEN LOWER(a_subject) LIKE 'email: re:%'                THEN 'email_reply'
            WHEN LOWER(a_subject) LIKE 'email:%'                    THEN 'email_sent'
            WHEN LOWER(a_subject) LIKE 'meeting:%'                  THEN 'meeting'
            WHEN LOWER(a_subject) LIKE 'demo:%'                     THEN 'demo'
            WHEN LOWER(a_subject) LIKE 'follow%up%'                 THEN 'follow_up'
            WHEN LOWER(a_subject) LIKE 'call:%'                     THEN 'call'
            WHEN LOWER(a_subject) LIKE 'webinar%'                   THEN 'webinar'
            WHEN LOWER(a_subject) LIKE 'linkedin%'                  THEN 'linkedin_outreach'
            WHEN LOWER(a_subject) LIKE 'signup%'                    THEN 'signup'
            WHEN LOWER(a_subject) LIKE '%sign up%'                  THEN 'signup'
            WHEN LOWER(a_subject) LIKE '%conversion%'               THEN 'conversion'
            WHEN LOWER(a_subject) LIKE '%closed won%'               THEN 'closed_won'
            WHEN LOWER(a_subject) LIKE '%closed lost%'              THEN 'closed_lost'
            WHEN a_subject IS NULL                                  THEN 'unknown_task'
            ELSE LOWER(REGEXP_REPLACE(a_subject, '[^a-zA-Z0-9]+', '_'))
        END
    """
    
    if target_count == 0:
        # First run: full load from source
        print(f"  First run detected — loading all tasks for tenant {tenant_id}")
        source_sql = f"""
            SELECT
                id, tenant, event_timestamp,
                CONCAT('salesforce_', COALESCE(contact_source_system_object, 'Contact'),
                       '_Id_', contact_source_key_value) AS contact_id,
                {meta_event_case} AS meta_event,
                a_subject AS event_text,
                data_timestamp
            FROM {sv}.tasks
            WHERE tenant = {tenant_id}
              AND contact_source_key_value IS NOT NULL
        """
        source_df = spark.sql(source_sql)
        cdf_count = source_df.count()
    elif last_source_ver >= source_max_ver:
        # No new versions on source — nothing to process
        print(f"  Source table unchanged (ver {source_max_ver}) — skipping")
        cdf_count = 0
        source_df = None
    else:
        # Incremental: read only CDF changes from silver.tasks
        start_ver = last_source_ver + 1
        try:
            cdf_df = (spark.read
                .format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.tasks")
                .filter(f"tenant = {tenant_id}")
                .filter("_change_type IN ('insert', 'update_postimage')")
                .filter("contact_source_key_value IS NOT NULL")
            )
            cdf_count = cdf_df.count()
            print(f"  CDF records found: {cdf_count} (versions {start_ver}..{source_max_ver})")
            
            if cdf_count == 0:
                print(f"  No changes detected — skipping MERGE")
                source_df = None
            else:
                # Register temp view and apply transformation
                cdf_df.createOrReplaceTempView("tasks_cdf_raw")
                source_df = spark.sql(f"""
                    SELECT
                        id, tenant, event_timestamp,
                        CONCAT('salesforce_', COALESCE(contact_source_system_object, 'Contact'),
                               '_Id_', contact_source_key_value) AS contact_id,
                        {meta_event_case} AS meta_event,
                        a_subject AS event_text,
                        data_timestamp
                    FROM tasks_cdf_raw
                """)
        except Exception as cdf_err:
            # CDF not available — fall back to watermark-based filtering
            print(f"  CDF read failed ({cdf_err}), falling back to watermark-based filter")
            last_ts = spark.sql(f"""
                SELECT COALESCE(MAX(data_timestamp), TIMESTAMP('1900-01-01'))
                FROM {sv}.crm_events WHERE tenant = {tenant_id}
            """).collect()[0][0]
            source_df = spark.sql(f"""
                SELECT
                    id, tenant, event_timestamp,
                    CONCAT('salesforce_', COALESCE(contact_source_system_object, 'Contact'),
                           '_Id_', contact_source_key_value) AS contact_id,
                    {meta_event_case} AS meta_event,
                    a_subject AS event_text,
                    data_timestamp
                FROM {sv}.tasks
                WHERE tenant = {tenant_id}
                  AND contact_source_key_value IS NOT NULL
                  AND data_timestamp > '{last_ts}'
            """)
            cdf_count = source_df.count()
            print(f"  Watermark-based records found: {cdf_count}")

    # Only run MERGE if we have changes
    if source_df is not None and cdf_count > 0:
        source_df.createOrReplaceTempView("crm_events_cdf_source")
        
        spark.sql(f"""
            MERGE INTO {sv}.crm_events AS target
            USING crm_events_cdf_source AS source
            ON target.id = source.id AND target.tenant = source.tenant
            WHEN MATCHED AND source.data_timestamp > target.data_timestamp THEN UPDATE SET
                target.event_timestamp = source.event_timestamp,
                target.contact_id = source.contact_id,
                target.meta_event = source.meta_event,
                target.event_text = source.event_text,
                target.data_timestamp = source.data_timestamp
            WHEN NOT MATCHED THEN INSERT (id, tenant, event_timestamp, contact_id, meta_event, event_text, data_timestamp)
                VALUES (source.id, source.tenant, source.event_timestamp, source.contact_id, 
                        source.meta_event, source.event_text, source.data_timestamp)
        """)
        print(f"  MERGE complete: {cdf_count} CDF records processed")
    else:
        print(f"  No changes to process")

    # ── Save the source version we just processed ──
    spark.sql(f"ALTER TABLE {sv}.crm_events SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

except Exception as e:
    print(f"❌ crm_events build failed: {e}")
    log_audit(
        job_name       = "02b_map_03_crm_events",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 5 ── Spec DQ verification
try:
    total     = cnt(f"{sv}.crm_events")
    null_meta = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.crm_events WHERE meta_event IS NULL"
    ).collect()[0][0]
    null_contact = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.crm_events WHERE contact_id IS NULL"
    ).collect()[0][0]

    print(f"  crm_events: {total:,} rows")
    print(f"  null meta_event : {null_meta}   (spec gate = 0, target = 100% non-null)")
    print(f"  null contact_id : {null_contact}  (these were filtered by contact_source_key_value IS NOT NULL)")

    # Show distribution of meta_event types for visibility
    print("\n  meta_event distribution:")
    spark.sql(f"""
        SELECT meta_event, COUNT(*) AS cnt
        FROM {sv}.crm_events
        GROUP BY meta_event
        ORDER BY cnt DESC
        LIMIT 10
    """).show(truncate=False)
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_03_crm_events",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise